In [1]:
from data.OPPORTUNITY_data import load_OPP_loco_data, data_split_OPP, make_loaders_OPP
from data.data_pipeline import load_opportunity
import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from data.data_pipeline import (load_opportunity, remove_zero_label_rows, incomplete_labeled_rows, remove_all_nan_rows, sliding_window, divide_features_labels, acc_data_scaling, gyro_data_scaling, mag_data_norm, mag_data_rotation)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from data.preprocessing import fit_labelencoder, Dataset_HAR
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import torch

In [2]:
training_files, validation_files, test_files = data_split_OPP(1)

In [3]:
training_data = load_opportunity(training_files, add_group_id=False)
validation_data = load_opportunity(validation_files, add_group_id=True)
test_data = load_opportunity(test_files, add_group_id=True)

In [4]:
training_data.head(2)

,0,1,2,3,4,5,6,7,8,9,...,240,241,242,243,244,245,246,247,248,249
0,0,-25.0,1051.0,-32.0,98.0,951.0,387.0,-482.0,386.0,764.0,...,4584.0,1911.0,1486.0,0,0,0,0,0,0,0
1,33,-62.0,1052.0,-6.0,76.0,968.0,374.0,-484.0,392.0,783.0,...,4582.0,1913.0,1487.0,0,0,0,0,0,0,0


In [5]:
validation_data.head(2)

,0,1,2,3,4,5,6,7,8,9,...,241,242,243,244,245,246,247,248,249,group_id
0,0,-70.0,1018.0,-121.0,89.0,976.0,-285.0,30.0,942.0,373.0,...,2094.0,1733.0,0,0,0,0,0,0,0,S2-ADL5
1,33,-74.0,1011.0,-125.0,88.0,957.0,-283.0,45.0,950.0,370.0,...,2089.0,1719.0,0,0,0,0,0,0,0,S2-ADL5


In [6]:
test_data.head(2)

,0,1,2,3,4,5,6,7,8,9,...,241,242,243,244,245,246,247,248,249,group_id
0,0,87.0,975.0,-287.0,11.0,1001.0,163.0,95.0,975.0,152.0,...,2907.0,1447.0,0,0,0,0,0,0,0,S1-ADL1
1,33,124.0,978.0,-389.0,-7.0,1014.0,199.0,124.0,968.0,123.0,...,2908.0,1443.0,0,0,0,0,0,0,0,S1-ADL1


In [7]:
imu_columns = list(range(37,46))+ list(range(50,59)) + list(range(63,72)) + list(range(76,85)) + list(range(89,98))
imu_columns.append(243) #labels
training_data_selected = training_data.iloc[:, imu_columns]
validation_data_selected = validation_data.iloc[:, imu_columns].copy()
validation_data_selected["group_id"] = validation_data["group_id"].values
test_data_selected = test_data.iloc[:, imu_columns].copy()
test_data_selected["group_id"] = test_data["group_id"].values

In [8]:
training_data_selected.head(2)

,37,38,39,40,41,42,43,44,45,50,...,89,90,91,92,93,94,95,96,97,243
0,-1008.0,-68.0,-26.0,-11.0,-47.0,22.0,316.0,-510.0,-243.0,-965.0,...,-810.0,-842.0,211.0,-1491.0,-361.0,-957.0,542.0,-81.0,465.0,0
1,-996.0,-58.0,-26.0,2.0,-35.0,22.0,316.0,-510.0,-243.0,-988.0,...,-772.0,-831.0,146.0,-1843.0,-376.0,-1378.0,553.0,-79.0,453.0,0


In [9]:
validation_data_selected.head(2)

,37,38,39,40,41,42,43,44,45,50,...,90,91,92,93,94,95,96,97,243,group_id
0,-981.0,-217.0,-39.0,123.0,46.0,25.0,420.0,-469.0,-194.0,-984.0,...,-539.0,99.0,-14.0,-263.0,431.0,488.0,7.0,546.0,0,S2-ADL5
1,-980.0,-220.0,-38.0,155.0,42.0,19.0,419.0,-471.0,-192.0,-992.0,...,-547.0,96.0,30.0,-294.0,535.0,494.0,1.0,542.0,0,S2-ADL5


In [10]:
test_data_selected.head(2)

,37,38,39,40,41,42,43,44,45,50,...,90,91,92,93,94,95,96,97,243,group_id
0,-983.0,-199.0,119.0,-166.0,-50.0,70.0,483.0,-499.0,42.0,-1016.0,...,-358.0,171.0,-145.0,47.0,48.0,389.0,202.0,593.0,0,S1-ADL1
1,-986.0,-220.0,114.0,-170.0,-31.0,103.0,479.0,-500.0,40.0,-1010.0,...,-357.0,161.0,-318.0,-6.0,23.0,390.0,196.0,597.0,0,S1-ADL1


In [11]:
training_data_nan = remove_zero_label_rows(training_data_selected)
validation_data_nan = remove_zero_label_rows(validation_data_selected)
test_data_nan = remove_zero_label_rows(test_data_selected)

In [12]:
training_data_nan.head(2)

,37,38,39,40,41,42,43,44,45,50,...,89,90,91,92,93,94,95,96,97,243
2395,-999.0,-86.0,-1.0,84.0,-4.0,35.0,313.0,-545.0,-198.0,-966.0,...,-856.0,-507.0,139.0,144.0,83.0,17.0,474.0,-26.0,549.0,1
2396,-1007.0,-66.0,-5.0,17.0,-7.0,9.0,313.0,-544.0,-194.0,-976.0,...,-858.0,-498.0,130.0,29.0,94.0,78.0,473.0,-22.0,547.0,1


In [13]:
validation_data_nan.head(2)

,37,38,39,40,41,42,43,44,45,50,...,90,91,92,93,94,95,96,97,243,group_id
2070,-968.0,-217.0,-6.0,365.0,51.0,128.0,464.0,-493.0,-83.0,-939.0,...,-553.0,56.0,124.0,176.0,260.0,562.0,-1.0,510.0,2,S2-ADL5
2071,-1010.0,-180.0,-6.0,491.0,80.0,124.0,463.0,-494.0,-77.0,-950.0,...,-567.0,47.0,304.0,260.0,298.0,560.0,1.0,515.0,2,S2-ADL5


In [14]:
test_data_nan.head(2)

,37,38,39,40,41,42,43,44,45,50,...,90,91,92,93,94,95,96,97,243,group_id
2954,-987.0,-130.0,106.0,-15.0,-16.0,-19.0,485.0,-508.0,202.0,-973.0,...,-359.0,129.0,-49.0,49.0,15.0,365.0,309.0,565.0,1,S1-ADL1
2955,-986.0,-128.0,106.0,9.0,1.0,-25.0,485.0,-508.0,204.0,-972.0,...,-366.0,134.0,-49.0,71.0,38.0,366.0,309.0,568.0,1,S1-ADL1


In [15]:
training_data_nan.columns = range(training_data_nan.shape[1])
training_data_nan = training_data_nan.reset_index(drop=True)

validation_data_nan.columns = list(range(validation_data_nan.shape[1] - 1)) + [validation_data_nan.columns[-1]]
validation_data_nan = validation_data_nan.reset_index(drop=True)

test_data_nan.columns = list(range(test_data_nan.shape[1] - 1)) + [test_data_nan.columns[-1]]
test_data_nan = test_data_nan.reset_index(drop=True)

In [16]:
training_data_nan.head(2)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,-999.0,-86.0,-1.0,84.0,-4.0,35.0,313.0,-545.0,-198.0,-966.0,...,-856.0,-507.0,139.0,144.0,83.0,17.0,474.0,-26.0,549.0,1
1,-1007.0,-66.0,-5.0,17.0,-7.0,9.0,313.0,-544.0,-194.0,-976.0,...,-858.0,-498.0,130.0,29.0,94.0,78.0,473.0,-22.0,547.0,1


In [17]:
validation_data_nan.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-968.0,-217.0,-6.0,365.0,51.0,128.0,464.0,-493.0,-83.0,-939.0,...,-553.0,56.0,124.0,176.0,260.0,562.0,-1.0,510.0,2,S2-ADL5
1,-1010.0,-180.0,-6.0,491.0,80.0,124.0,463.0,-494.0,-77.0,-950.0,...,-567.0,47.0,304.0,260.0,298.0,560.0,1.0,515.0,2,S2-ADL5


In [18]:
test_data_nan.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-987.0,-130.0,106.0,-15.0,-16.0,-19.0,485.0,-508.0,202.0,-973.0,...,-359.0,129.0,-49.0,49.0,15.0,365.0,309.0,565.0,1,S1-ADL1
1,-986.0,-128.0,106.0,9.0,1.0,-25.0,485.0,-508.0,204.0,-972.0,...,-366.0,134.0,-49.0,71.0,38.0,366.0,309.0,568.0,1,S1-ADL1


In [19]:
training_data_cleaned = remove_all_nan_rows(training_data_nan)
validation_data_cleaned = remove_all_nan_rows(validation_data_nan)
test_data_cleaned = remove_all_nan_rows(test_data_nan)

In [20]:
training_data_cleaned.head(2)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,-999.0,-86.0,-1.0,84.0,-4.0,35.0,313.0,-545.0,-198.0,-966.0,...,-856.0,-507.0,139.0,144.0,83.0,17.0,474.0,-26.0,549.0,1
1,-1007.0,-66.0,-5.0,17.0,-7.0,9.0,313.0,-544.0,-194.0,-976.0,...,-858.0,-498.0,130.0,29.0,94.0,78.0,473.0,-22.0,547.0,1


In [21]:
validation_data_cleaned.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-968.0,-217.0,-6.0,365.0,51.0,128.0,464.0,-493.0,-83.0,-939.0,...,-553.0,56.0,124.0,176.0,260.0,562.0,-1.0,510.0,2,S2-ADL5
1,-1010.0,-180.0,-6.0,491.0,80.0,124.0,463.0,-494.0,-77.0,-950.0,...,-567.0,47.0,304.0,260.0,298.0,560.0,1.0,515.0,2,S2-ADL5


In [22]:
training_data_scaled = acc_data_scaling(training_data_cleaned)
validation_data_scaled = acc_data_scaling(validation_data_cleaned)
test_data_scaled = acc_data_scaling(test_data_cleaned)

In [23]:
training_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,-0.999,-0.086,-0.001,84.0,-4.0,35.0,313.0,-545.0,-198.0,-0.966,...,-0.856,-0.507,0.139,144.0,83.0,17.0,474.0,-26.0,549.0,1
1,-1.007,-0.066,-0.005,17.0,-7.0,9.0,313.0,-544.0,-194.0,-0.976,...,-0.858,-0.498,0.130,29.0,94.0,78.0,473.0,-22.0,547.0,1


In [24]:
validation_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.968,-0.217,-0.006,365.0,51.0,128.0,464.0,-493.0,-83.0,-0.939,...,-0.553,0.056,124.0,176.0,260.0,562.0,-1.0,510.0,2,S2-ADL5
1,-1.010,-0.180,-0.006,491.0,80.0,124.0,463.0,-494.0,-77.0,-0.950,...,-0.567,0.047,304.0,260.0,298.0,560.0,1.0,515.0,2,S2-ADL5


In [25]:
test_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.987,-0.130,0.106,-15.0,-16.0,-19.0,485.0,-508.0,202.0,-0.973,...,-0.359,0.129,-49.0,49.0,15.0,365.0,309.0,565.0,1,S1-ADL1
1,-0.986,-0.128,0.106,9.0,1.0,-25.0,485.0,-508.0,204.0,-0.972,...,-0.366,0.134,-49.0,71.0,38.0,366.0,309.0,568.0,1,S1-ADL1


In [26]:
training_data_scaled = gyro_data_scaling(training_data_scaled)
validation_data_scaled = gyro_data_scaling(validation_data_scaled)
test_data_scaled = gyro_data_scaling(test_data_scaled)
training_data_scaled = mag_data_norm(training_data_scaled)
validation_data_scaled = mag_data_norm(validation_data_scaled)
test_data_scaled = mag_data_norm(test_data_scaled)

In [27]:
training_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,-0.999,-0.086,-0.001,0.084,-0.004,0.035,0.475007,-0.82709,-0.300484,-0.966,...,-0.856,-0.507,0.139,0.144,0.083,0.017,0.653093,-0.035824,0.756430,1
1,-1.007,-0.066,-0.005,0.017,-0.007,0.009,0.476468,-0.82811,-0.295319,-0.976,...,-0.858,-0.498,0.130,0.029,0.094,0.078,0.653785,-0.030409,0.756069,1


In [28]:
validation_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.968,-0.217,-0.006,0.365,0.051,0.128,0.680271,-0.722788,-0.121686,-0.939,...,-0.553,0.056,0.124,0.176,0.260,0.740535,-0.001318,0.672016,2,S2-ADL5
1,-1.010,-0.180,-0.006,0.491,0.080,0.124,0.679463,-0.724956,-0.112999,-0.950,...,-0.567,0.047,0.304,0.260,0.298,0.736061,0.001314,0.676914,2,S2-ADL5


In [29]:
test_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.987,-0.130,0.106,-0.015,-0.016,-0.019,0.663641,-0.695113,0.276403,-0.973,...,-0.359,0.129,-0.049,0.049,0.015,0.493094,0.417441,0.763283,1,S1-ADL1
1,-0.986,-0.128,0.106,0.009,0.001,-0.025,0.663137,-0.694585,0.278928,-0.972,...,-0.366,0.134,-0.049,0.071,0.038,0.492592,0.415877,0.764460,1,S1-ADL1


In [30]:
scaler = StandardScaler()
acc_gyro_cols = [axis + offset for offset in range(0, 45, 9) for axis in range(6)]
training_data_scaled.iloc[:, acc_gyro_cols] = scaler.fit_transform(training_data_scaled.iloc[:, acc_gyro_cols].values)
validation_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(validation_data_scaled.iloc[:, acc_gyro_cols].values)
test_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(test_data_scaled.iloc[:, acc_gyro_cols].values)

In [31]:
training_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,-0.650737,0.135490,-0.971640,0.212024,0.037801,0.098677,0.475007,-0.82709,-0.300484,-0.597862,...,-0.775936,0.349734,0.751200,0.301802,0.106985,-0.013179,0.653093,-0.035824,0.756430,1
1,-0.680319,0.215497,-0.984363,0.100312,0.028506,0.001237,0.476468,-0.82811,-0.295319,-0.634684,...,-0.779670,0.380858,0.724873,0.151938,0.122089,0.071190,0.653785,-0.030409,0.756069,1


In [32]:
validation_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.536108,-0.388554,-0.987543,0.680546,0.208200,0.447212,0.680271,-0.722788,-0.121686,-0.498444,...,0.190658,0.508405,0.275738,0.234687,0.322912,0.740535,-0.001318,0.672016,2,S2-ADL5
1,-0.691412,-0.240541,-0.987543,0.890630,0.298047,0.432221,0.679463,-0.724956,-0.112999,-0.538948,...,0.142244,0.482078,0.510308,0.350031,0.375470,0.736061,0.001314,0.676914,2,S2-ADL5


In [33]:
test_data_scaled.head(2)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,group_id
0,-0.606365,-0.040525,-0.631311,0.046958,0.000623,-0.103698,0.663641,-0.695113,0.276403,-0.623637,...,0.861545,0.721948,0.050291,0.060298,-0.015945,0.493094,0.417441,0.763283,1,S1-ADL1
1,-0.602667,-0.032524,-0.631311,0.086974,0.053292,-0.126184,0.663137,-0.694585,0.278928,-0.619955,...,0.837338,0.736574,0.050291,0.090507,0.015866,0.492592,0.415877,0.764460,1,S1-ADL1


In [47]:
combined_eval = pd.concat([validation_data_scaled, test_data_scaled], axis = 0, ignore_index = True) 
y = combined_eval.iloc[:, -2].values       # Labels
groups = combined_eval["group_id"].values  # File id Name
    
sgkf = StratifiedGroupKFold(n_splits = 2, shuffle = True, random_state = 42)
val_idx, test_idx = next(sgkf.split(combined_eval, y, groups))
print(f"{'-'*20}STRATIY VAL/TEST{'-'*20}")
print("Validation GROUPS")
print(combined_eval.iloc[val_idx]["group_id"].value_counts().sort_index())
print("Test GROUPS")
print(combined_eval.iloc[test_idx]["group_id"].value_counts().sort_index(), "\n")

#val_with_groups = combined_eval.iloc[val_idx].copy()
#test_with_groups = combined_eval.iloc[test_idx].copy()
#print("Validation Split label proportion within each group")
#print(pd.crosstab(val_with_groups["group_id"], val_with_groups.iloc[:, -2], normalize="index"))
#print("Test Split label proportion within each group")
#print(pd.crosstab(test_with_groups["group_id"], test_with_groups.iloc[:, -2], normalize="index"))

new_validation_data_scaled = combined_eval.iloc[val_idx].drop(columns=["group_id"]).copy()
new_test_data_scaled = combined_eval.iloc[test_idx].drop(columns=["group_id"]).copy()
print(f"\n Validation Split Label Proportion")
print(new_validation_data_scaled.iloc[:, -1].value_counts(normalize=True).sort_index())
print("Test Split Label Proportion")
print(new_test_data_scaled.iloc[:, -1].value_counts(normalize=True).sort_index())

--------------------STRATIY VAL/TEST--------------------
Validation GROUPS
group_id
S1-ADL4    24851
S2-ADL5    22969
S3-ADL5    20648
S4-ADL5    23817
Name: count, dtype: int64
Test GROUPS
group_id
S1-ADL1    37507
S1-ADL2    24510
S1-ADL3    25305
S1-ADL5    22440
Name: count, dtype: int64 


 Validation Split Label Proportion
45
1    0.435575
2    0.285561
4    0.229149
5    0.049716
Name: proportion, dtype: float64
Test Split Label Proportion
45
1    0.476476
2    0.221643
4    0.256008
5    0.045872
Name: proportion, dtype: float64
